# Tomato Leaf Dataset - Scaling Law Analysis (Colab Version)

Este notebook lê o CSV de métricas gerado durante o treinamento (`training_results.csv`), calcula as médias das métricas na última época de cada run, ajusta uma curva de **Regressão Logarítmica (Scaling Law)** baseada nos primeiros subsets (ex: 1%, 2% e 5%) e tenta prever algebricamente o comportamento nos próximos subsets (ex: 10%, 20%, 50%, 100%), plotando um gráfico comparativo inline.

In [ ]:
# --- CONFIGURAÇÕES DE CAMINHO ---
# Se estiver usando o Google Drive, monte o drive e aponte para o caminho correto, ex:
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = "/content/drive/MyDrive/redes_neurais/training_results.csv"

CSV_PATH = "training_results.csv"
FIT_POINTS = 3  # Quantidade de subsets iniciais usados para ajustar a curva de previsão (ex: 3 usa 1%, 2% e 5%)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def fit_logarithmic_curve(x, y):
    """
    Ajusta y = a * ln(x) + b usando mínimos quadrados.
    """
    log_x = np.log(x)
    a, b = np.polyfit(log_x, y, 1)
    return a, b

def predict_logarithmic(x_pred, a, b):
    """
    Prediz o valor de y com base no x e nos coeficientes a e b.
    """
    return a * np.log(x_pred) + b

try:
    csv_file = Path(CSV_PATH)
    if not csv_file.exists():
        raise FileNotFoundError(f"Arquivo CSV em '{CSV_PATH}' não encontrado. Verifique se o treino já foi rodado.")
        
    # 1. Carrega e limpa o DataFrame
    df = pd.read_csv(csv_file)
    
    # Filtra para obter apenas a última época de cada run
    idx = df.groupby(["subset", "run_index"])["epoch"].idxmax()
    df_final = df.loc[idx].copy()
    
    # Extrai a porcentagem numérica
    df_final["percentage"] = df_final["subset"].str.extract(r"(\d+)").astype(float)
    
    # Agrupa por porcentagem e tira a média entre os runs (seeds)
    agg_df = df_final.groupby("percentage").agg({
        "train_loss": "mean",
        "test_accuracy": "mean"
    }).reset_index().sort_values("percentage")
    
    percentages = agg_df["percentage"].values
    losses = agg_df["train_loss"].values
    accuracies = agg_df["test_accuracy"].values
    
    print("=== Métricas Médias por Subset (Última Época) ===")
    for idx, row in agg_df.iterrows():
        print(f"  Subset {row['percentage']:>5.1f}% | Loss: {row['train_loss']:.4f} | Acurácia: {row['test_accuracy']:.4f}")
        
    if len(percentages) < FIT_POINTS:
        raise ValueError(f"Número insuficiente de subsets no CSV ({len(percentages)}) para ajustar com FIT_POINTS={FIT_POINTS}.")

    # 2. Executa previsões sequenciais a partir do índice de FIT_POINTS
    print("\n=== Previsões Sequenciais de Scaling Law ===")
    print(f"{'Subset Alvo':<12} | {'Métrica':<10} | {'Real':<10} | {'Previsto':<10} | {'Erro Abs':<10} | {'Erro Rel %':<10}")
    print("-" * 75)
    
    for k in range(FIT_POINTS, len(percentages)):
        target_pct = percentages[k]
        x_train = percentages[:k]
        
        # Predição de Loss
        y_train_loss = losses[:k]
        a_loss, b_loss = fit_logarithmic_curve(x_train, y_train_loss)
        pred_loss = predict_logarithmic(target_pct, a_loss, b_loss)
        act_loss = losses[k]
        abs_err_loss = abs(act_loss - pred_loss)
        rel_err_loss = (abs_err_loss / act_loss) * 100 if act_loss != 0 else 0.0
        print(f"{target_pct:>10.1f}% | {'Loss':<10} | {act_loss:>10.4f} | {pred_loss:>10.4f} | {abs_err_loss:>10.4f} | {rel_err_loss:>9.2f}%")
        
        # Predição de Acurácia
        y_train_acc = accuracies[:k]
        a_acc, b_acc = fit_logarithmic_curve(x_train, y_train_acc)
        pred_acc = predict_logarithmic(target_pct, a_acc, b_acc)
        pred_acc_clipped = np.clip(pred_acc, 0.0, 1.0)
        act_acc = accuracies[k]
        abs_err_acc = abs(act_acc - pred_acc_clipped)
        rel_err_acc = (abs_err_acc / act_acc) * 100 if act_acc != 0 else 0.0
        print(f"{'':<12} | {'Acurácia':<10} | {act_acc:>10.4f} | {pred_acc_clipped:>10.4f} | {abs_err_acc:>10.4f} | {rel_err_acc:>9.2f}%")
        print("-" * 75)
        
    # 3. Ajusta a curva geral com os primeiros FIT_POINTS e plota
    x_fit = percentages[:FIT_POINTS]
    y_fit_loss = losses[:FIT_POINTS]
    y_fit_acc = accuracies[:FIT_POINTS]
    
    a_loss, b_loss = fit_logarithmic_curve(x_fit, y_fit_loss)
    a_acc, b_acc = fit_logarithmic_curve(x_fit, y_fit_acc)
    
    x_curve = np.linspace(percentages[0], percentages[-1], 200)
    y_curve_loss = predict_logarithmic(x_curve, a_loss, b_loss)
    y_curve_acc = np.clip(predict_logarithmic(x_curve, a_acc, b_acc), 0.0, 1.0)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"Análise de Scaling Law (Regressão Logarítmica fitted nos {FIT_POINTS} primeiros subsets)", fontsize=14, fontweight='bold')
    
    # Plot de Loss
    ax1.scatter(percentages, losses, color="red", label="Métrica Real (Média)", s=60, zorder=5)
    ax1.plot(x_curve, y_curve_loss, color="salmon", linestyle="--", label="Curva Logarítmica Prevista")
    ax1.scatter(x_fit, y_fit_loss, facecolors='none', edgecolors='black', s=150, linewidths=2, label="Bases do Ajuste", zorder=6)
    ax1.set_xscale("log")
    ax1.set_xlabel("Porcentagem do Dataset (Escala Log)")
    ax1.set_ylabel("Train Loss")
    ax1.set_title("Treino: Loss vs. Escala do Dataset")
    ax1.grid(True, which="both", linestyle=":", alpha=0.5)
    ax1.legend()
    
    # Plot de Acurácia
    ax2.scatter(percentages, accuracies, color="blue", label="Métrica Real (Média)", s=60, zorder=5)
    ax2.plot(x_curve, y_curve_acc, color="skyblue", linestyle="--", label="Curva Logarítmica Prevista")
    ax2.scatter(x_fit, y_fit_acc, facecolors='none', edgecolors='black', s=150, linewidths=2, label="Bases do Ajuste", zorder=6)
    ax2.set_xscale("log")
    ax2.set_xlabel("Porcentagem do Dataset (Escala Log)")
    ax2.set_ylabel("Test Accuracy (Ratio)")
    ax2.set_title("Teste: Acurácia vs. Escala do Dataset")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", linestyle=":", alpha=0.5)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"\n[ERRO] Ocorreu uma falha na análise: {e}")